In [1]:
%pip install requests pandas openpyxl argparse re threading
import getpass
import argparse
import json
import os
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests

  Using cached argparse-1.4.0-py2.py3-none-any.whl.metadata (2.8 kB)
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement re (from versions: none)

[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for re


# STEP 1: Settings

In [ ]:
HOST  = "https://maki.uni-mannheim.de/v1"
MODEL = "gemma4-26b"
TEMPERATURE = 0           # 0 = deterministic / predictable
SEED = 123
MAX_WORKERS = 2              # Parallel calls; lower it if the server returns errors
TIMEOUT = 180                # Seconds to wait for one answer
MAX_RETRIES = 3              # Tries per paragraph before it is marked as an error
STOP_AFTER_ERRORS = 20       # Pause the whole run if this many fail in a row

# Prompt for API Key
API_KEY = os.environ.get("UNI_API_KEY") or getpass.getpass("Paste the Uni API key: ")
print("Key loaded:", bool(API_KEY))

Key loaded: True


# STEP 2: The Annotation Prompt

In [ ]:
if "API_KEY" not in globals():
    raise RuntimeError("Please run STEP 1 first.")

# Customize your labels and definition here
ANNOTATION_PROMPT = """

ROLE
You are a text annotator specializing in media content analysis and computational communication. Your task is to apply the Stereotype Content Model (SCM) to evaluate whether targeted social groups in translated German newspaper articles are portrayed with Warmth.

TASK
Based on the multiple paragraphs provided, decide whether [Gruppe 1] (or an individual member of [Gruppe 1]) is attributed with warmth. The label is TRUE if the text contains warmth and FALSE if it does not. Prioritize explicit mentions of warm attributes. Implicit portrayals are also encouraged to be coded when the text clearly conveys warmth without using a trait word. Give a short explanation of why you chose TRUE or FALSE, based on the criteria below. While coding, target only [Gruppe 1]. Do NOT code whether other groups ([andere Gruppe]) display warmth toward [Gruppe 1].

CONTEXT
You are working with paragraphs from German newspaper articles from a wide range of the political spectrum. The paragraphs have been translated into English. Each input consists of several consecutive paragraphs from one article, together with the target group [Gruppe 1]. Return one label for the whole passage. Entity names in the text are masked as [Gruppe 1] and [andere Gruppe]. Keep these names exactly as written and do not translate, change or replace them, including in your explanation.

DEFINITION
Warmth in news coverage is defined as an individual's or group's perceived positive intention towards the perceiver or their group of belonging. Warmth is conceptualized through two dimensions in media reporting:

Friendliness/sociability: Framing the individual or group as friendly if they are depicted as someone who treats others in a friendly, kind and helpful manner.
Attribute markers include but are not limited to: sociable, friendly, warm, liked, outgoing, likeable, pleasant, popular

Morality: Framing the individual or group as moral if they are attributed as someone trustworthy, honest, fair and good.
Attribute markers include but are not limited to: moral, trustworthy, sincere, fair, tolerant, altruistic, benevolent

CRITERIA: WARMTH OCCURS IF

Case: A warm trait is attributed to [Gruppe 1] or a member.
Example: "The [Gruppe 1] are considered welcoming / hospitable."

Case: A synonym or noun form of a warm trait is used.
Example: "Their honesty is praised."

Case: A behaviour that clearly shows good intentions toward others.
Example: "The [Gruppe 1] native collected donations for the refugees."

Case: A positive moral evaluation.
Example: "The [Gruppe 1] fight heroically."

CRITERIA: WARMTH DOES NOT OCCUR IF

Case: Others are warm TOWARD the group.
Example: "The local population showed great readiness to help the [Gruppe 1]." / "The public showed great willingness to assist [Gruppe 1]."
Why not: The warmth belongs to the Germans, not to [Gruppe 1].

Case: The group is only a victim.
Example: "Three [Gruppe 1] were attacked."
Why not: Being a victim is not a trait.

Case: It is about ability or courage (competence).
Example: "The brave [Gruppe 1]", "hard-working"
Why not: Courage, diligence and skill = competence.

Case: The warm word describes someone else.
Example: "Filip takes care of everyone," says the [Gruppe 1].
Why not: [Gruppe 1] is only the speaker.

Case: It is only a benefit frame.
Example: "[Gruppe 1] strengthen the economy."
Why not: An economic benefit, not a trait (Frames dimension).

Case: No portrayal at all.
Example: "In 2022, 12,000 [Gruppe 1] applied for asylum."
Why not: Facts and statistics only.

DECISION RULES
tba.

OUTPUT FORMAT
Return only a valid JSON object, with no text before or after it. The object has exactly two keys, in this order:
"explanation": 1-2 sentences explaining the label, citing the specific words or behavior that led to it.
"label": the string "TRUE" or "FALSE".

Example of valid output:
{"explanation": "The text describes the [Gruppe 1] as hospitable and welcoming toward newcomers.", "label": "TRUE"}

"""

instructions = ANNOTATION_PROMPT
reminder = None  # this is just an additonal vairable to nudge the llm to a specific direction
json_mode= True

print("--- Instructions to annotate WARMTH Loaded ---")
print(instructions)

--- Instructions to annotate WARMTH Loaded ---


ROLE
You are a text annotator specializing in media content analysis and computational communication. Your task is to apply the Stereotype Content Model (SCM) to evaluate whether targeted social groups in translated German newspaper articles are portrayed with Warmth.

TASK
Based on the multiple paragraphs provided, decide whether [Gruppe 1] (or an individual member of [Gruppe 1]) is attributed with warmth. The label is TRUE if the text contains warmth and FALSE if it does not. Prioritize explicit mentions of warm attributes. Implicit portrayals are also encouraged to be coded when the text clearly conveys warmth without using a trait word. Give a short explanation of why you chose TRUE or FALSE, based on the criteria below. While coding, target only [Gruppe 1]. Do NOT code whether other groups ([andere Gruppe]) display warmth toward [Gruppe 1].

CONTEXT
You are working with paragraphs from German newspaper articles from a wide range of 

# STEP 3: Paragraph Input Data

In [4]:
if "API_KEY" not in globals():
    raise RuntimeError("Please run STEP 1 first.")

INPUT_EXCEL_FILE = "sample_5k_translated.xlsx"
CHECKPOINT_FILE = "warmth_annotation_checkpoint.jsonl"
OUTPUT_EXCEL = "sample_5k_warmth_annotated.xlsx"
OUTPUT_JSON = "sample_5k_warmth_annotated.json"

try:
    sample = pd.read_excel(INPUT_EXCEL_FILE)
    if "row_id" not in sample.columns:
        sample.insert(0, "row_id", range(len(sample)))
    print(f"Successfully loaded {len(sample)} paragraph(s) from '{INPUT_EXCEL_FILE}'.")
except FileNotFoundError:
    raise FileNotFoundError(f"Error: Could not find '{INPUT_EXCEL_FILE}' in the current folder.")


Successfully loaded 5000 paragraph(s) from 'sample_5k_translated.xlsx'.


# STEP 4: Set up LLM API Call Function

In [ ]:
if "API_KEY" not in globals():
    raise RuntimeError("Please run STEP 1 first.")

def call_llm(text, instructions, reminder=None, max_retries=MAX_RETRIES):
    user_msg = f"Text: '{text}'"
    if reminder:
        user_msg += f"\nReminder: {reminder}"
        
    body = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": instructions},
            {"role": "user",   "content": user_msg},
        ],
        "temperature": TEMPERATURE,
        "seed": SEED,
        "reasoning_effort": "none",
        "response_format": {"type": "json_object"},
    }
    
    last_error = None
    if json_mode:
        body["response_format"] = {"type": "json_object"}

        
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.post(f"{HOST}/chat/completions",
                              headers={"Authorization": f"Bearer {API_KEY}"},
                              json=body, timeout=TIMEOUT)
                                             
            r.raise_for_status()
            j = r.json()
            return {
                "output": j["choices"][0]["message"]["content"].strip(),
                "tokens_prompt": j["usage"]["prompt_tokens"],
                "tokens_response": j["usage"]["completion_tokens"],
                "error": None,
            }
            
        except requests.exceptions.HTTPError as he:
            last_error = str(he)
            if he.response.status_code == 504:
                # 504 Gateway Timeout: back off aggressively 
                wait_time = 15 * (2 ** (attempt - 1))
                print(f"Attempt {attempt}/{max_retries} failed: 504 Gateway Timeout. Server is overloaded. Waiting {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"Attempt {attempt}/{max_retries} failed with HTTP error: {he}")
                time.sleep(5 * attempt)
                # Back off softly if too many requests are sent

            if r.status_code == 429:
                                wait_time = 60
                                print(f"Attempt {attempt}/{max_retries} failed: 429 Too Many Requests. Cooling down for {wait_time}s...")
                                time.sleep(wait_time)
                                continue

            if he.response.status_code is not None and 400 <= he.response.status_code < 500 and he.response.status_code not in (408, 429):
                break

        except Exception as e:
            last_error = str(e)
            print(f"Attempt {attempt}/{max_retries} failed (Network error): {e}")
            time.sleep(5 * attempt)
            
    print("All retry attempts failed for this entry.")
    return {"output": None, "tokens_prompt": None, "tokens_response": None, "error": last_error}

# STEP 5: Run Annotation Loop with parallelization 

In [ ]:
if "API_KEY" not in globals():
    raise RuntimeError("Please run STEP 1 first.")

def annotate_row(row_id, text):
    start = time.time()
    res = call_llm(text)
    seconds = round(time.time() - start, 1)
    
    raw = res["output"]
    parsed_json = {}
    json_ok = False
    
    try:
        parsed_json = json.loads(raw)
        if isinstance(parsed_json, dict) and "label" in parsed_json and "explanation" in parsed_json:
            json_ok = True
    except Exception:
        json_ok = False

    return {
        "row_id": row_id,
        "label": parsed_json.get("label"),
        "explanation": parsed_json.get("explanation"),
        "raw_output": raw,
        "json_ok": json_ok,
        "tokens_prompt": res["tokens_prompt"],
        "tokens_response": res["tokens_response"],
        "seconds": seconds,
        "error": res["error"],
        "model": MODEL,
        "seed": SEED,
    }

####### Parallelization helper functions

def model_of(rec):
    return rec.get("model") or MODEL

def read_checkpoint():
    if not os.path.exists(CHECKPOINT_FILE):
        return []
    with open(CHECKPOINT_FILE, encoding="utf-8") as f:
        records = []
        for line in f:
            if line.strip():
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    continue
        return records

def done_ids():
    return {
        r["row_id"] for r in read_checkpoint()
        if r["error"] is None and r.get("label") in ["TRUE", "FALSE"] and model_of(r) == MODEL
    }

def save_checkpoint(rec):
    with _write_lock:
        with open(CHECKPOINT_FILE, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def run_parallel(rows, workers):
    done = done_ids()
    todo = rows[~rows["row_id"].isin(done)]
    print(f"To annotate: {len(todo)} | already done with {MODEL}: {len(rows) - len(todo)} | parallel workers: {workers}")
    if todo.empty:
        return True

    t0, n, n_err, in_a_row = time.time(), 0, 0, 0
    pool = ThreadPoolExecutor(max_workers=workers)
    
    # Here we annotate the english version and not the german one (which would be text)
    text_column = "translation" if "translation" in rows.columns else "text"
    
    futures = [pool.submit(annotate_row, r.row_id, getattr(r, text_column)) for r in todo.itertuples()]
    
    try:
        for fut in as_completed(futures):
            rec = fut.result()
            save_checkpoint(rec)
            n += 1
            n_err += rec["error"] is not None
            in_a_row = in_a_row + 1 if rec["error"] else 0
            
            if in_a_row >= STOP_AFTER_ERRORS:
                print(f"{in_a_row} errors in a row: server seems unresponsive. Paused.")
                return False
                
            if n % 50 == 0 or n == len(todo):
                el = time.time() - t0
                rate = n / (el / 60) if el > 0 else 0
                eta = (len(todo) - n) / rate if rate > 0 else 0
                print(f"{n}/{len(todo)} | errors {n_err} | {rate:.1f} per min | ~{eta:.0f} min left")
    finally:
        pool.shutdown(wait=False, cancel_futures=True)
    return True

Starting annotation loop for 5000 entries...

[1/5000] Paragraph 1 processed in 1.4s | JSON parsed: True
Annotations: {
  "explanation": "The text provides factual information regarding donations and transport logistics for [Gruppe 1] without attributing any specific personality traits, moral qualities, or friendly behaviors to them.",
  "label": "FALSE"
}
--------------------------------------------------
[2/5000] Paragraph 2 processed in 1.2s | JSON parsed: True
Annotations: {
  "explanation": "The text mentions the [Gruppe 1] only in the context of a political accusation regarding 'social tourism' and does not attribute any warm traits or behaviors to them.",
  "label": "FALSE"
}
--------------------------------------------------
[3/5000] Paragraph 3 processed in 1.1s | JSON parsed: True
Annotations: {
  "explanation": "The text does not attribute any warm traits, moral qualities, or friendly behaviors to [Gruppe 1]; it only mentions them in the context of a financial debt.",
  "lab

# STEP 6: Quality Checks & Verification

In [7]:
if "API_KEY" not in globals():
    raise RuntimeError("Please run STEP 1 first.")

valid_labels = {"TRUE", "FALSE"}

for idx, row in sample.iterrows():
    p_id = row.get("id", idx + 1)
    lbl = row.get("label")
    exp = str(row.get("explanation") or "")
    raw_text = str(row.get("text", ""))

    print(f"\n--- Quality Check [ID {p_id}] ---")
    
    # 1. Check JSON parsing
    json_ok = row.get("json_ok", False)
    print("JSON Valid:         ", "OK" if json_ok else "CHECK raw_output!")

    # 2. Check label validity
    label_ok = lbl in valid_labels
    print("Valid Label (T/F):  ", f"OK ({lbl})" if label_ok else f"CHECK! ({lbl})")

    # 3. Check explanation presence
    exp_ok = len(exp.strip()) > 0
    print("Explanation Present:", "OK" if exp_ok else "CHECK! (empty explanation)")

    # 4. Check entity masking preservation ([Gruppe 1])
    # Warns if [Gruppe 1] existed in input but model didn't keep it in explanation
    if "[Gruppe 1]" in raw_text:
        mask_ok = "[Gruppe 1]" in exp
        print("[Gruppe 1] Preserved:", "OK" if mask_ok else "CHECK! ([Gruppe 1] missing from explanation)")


--- Quality Check [ID 1] ---
JSON Valid:          OK
Valid Label (T/F):   OK (FALSE)
Explanation Present: OK

--- Quality Check [ID 2] ---
JSON Valid:          OK
Valid Label (T/F):   OK (FALSE)
Explanation Present: OK

--- Quality Check [ID 3] ---
JSON Valid:          OK
Valid Label (T/F):   OK (FALSE)
Explanation Present: OK

--- Quality Check [ID 4] ---
JSON Valid:          OK
Valid Label (T/F):   OK (FALSE)
Explanation Present: OK

--- Quality Check [ID 5] ---
JSON Valid:          OK
Valid Label (T/F):   OK (FALSE)
Explanation Present: OK

--- Quality Check [ID 6] ---
JSON Valid:          OK
Valid Label (T/F):   OK (FALSE)
Explanation Present: OK

--- Quality Check [ID 7] ---
JSON Valid:          OK
Valid Label (T/F):   OK (FALSE)
Explanation Present: OK

--- Quality Check [ID 8] ---
JSON Valid:          OK
Valid Label (T/F):   OK (FALSE)
Explanation Present: OK

--- Quality Check [ID 9] ---
JSON Valid:          OK
Valid Label (T/F):   OK (FALSE)
Explanation Present: OK

--- Quali

# STEP 7: Save the Results

In [ ]:
def save_results(sample):
    recs = [r for r in read_checkpoint() if model_of(r) == MODEL and r["error"] is None]
    res_df = pd.DataFrame(recs).drop_duplicates("row_id", keep="last")
    final = sample.merge(res_df, on="row_id", how="left")
    
    ok_count = final["label"].notna().sum()
    print(f"\nSuccessfully annotated with {MODEL}: {ok_count} / {len(final)}")
    
    # Run verification
    run_quality_checks(final)

    # Clean structured export for JSON
    json_export_data = []
    text_column = "translation" if "translation" in final.columns else "text"
    
    for _, row in final.iterrows():
        entry = {
            "id": int(row.get("row_id", 0)),
            "text": row.get(text_column),
            "warmth_annotation": {
                "label": row.get("label"),
                "explanation": row.get("explanation")
            },
            "metadata": {
                "json_parsed_ok": bool(row.get("json_ok", False)),
                "tokens_prompt": row.get("tokens_prompt"),
                "tokens_response": row.get("tokens_response"),
                "seconds": row.get("seconds")
            }
        }
        json_export_data.append(entry)

    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(json_export_data, f, ensure_ascii=False, indent=2)
        
    final.to_excel(OUTPUT_EXCEL, index=False)
    print(f"\nFinished! Saved annotations to '{OUTPUT_JSON}' and '{OUTPUT_EXCEL}'.")

 Finished! Saved clean annotations to 'warmth_annotation_results.json' and 'warmth_annotation_results.xlsx'.
